# 04 · Water saturation and net pay
**F-O gas field — Notebook 04** (input: unzipped **F-O_composites_v4**; output: composite **v5** + pay tables)

Inputs and where they come from
* **Rw** — synthetic formation brine of the F-O6 SCAL: 0.2678 ohm·m at 77 °F (~23 000 ppm NaCl), corrected to temperature (Arps).
  Checked against apparent Rw (Rwa = Rt·φ^m) in the water legs below the gas–water contact (Step 1).
* **Temperature** — linear geotherm through the F-O2 DST (309.7 °F at 3644 m).
* **Electrical parameters** — F-O2 SCAL (m 2.06, n 1.81; m* 2.2, n* 1.91) and F-O6 SCAL (m 1.79, n 2.65; m* 2.00, n* 2.88).
* **Clay conductivity** — F-O6 CEC (9 plugs, median ≈ 1 meq/100 g) does **not** correlate with log Vsh (r ≈ 0.3 / 0.0):
  the conductive clay is pore-lining (authigenic chlorite), so Qv is computed from a constant CEC and porosity, not from Vsh.
* **Gas–water contact** — field-wide 3794–3796 m TVDSS (F-O4: 3818 m MD). Depth datum (drill floor) above MSL from the tape
  headers: F-O1 26.0, F-O2 25.0, F-O3 22.0, F-S1 22.0, F-O6 25.0 (+1.45 m deviation), F-O8 25.0 m; F-O4 24 m (report);
  F-R1 22 m assumed (not on tape).

In [ ]:
# Cell 1 — paths (EDIT)
from pathlib import Path
V4_DIR  = Path(r"C:\Users\YOUR_NAME\...\F-O_composites_v4")
OUT_DIR = Path(r"C:\Users\YOUR_NAME\...\F-O_v5_outputs")
OUT_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
# Cell 2 — load
import numpy as np, pandas as pd, matplotlib.pyplot as plt, lasio, warnings
warnings.filterwarnings("ignore")
from petrotools import saturation as st, petromodels as pm
WELLS = ["F-O1", "F-O2", "F-O3", "F-O4", "F-O6", "F-O8", "F-R1", "F-S1"]
logs = {w: pd.read_csv(V4_DIR / f"{w}_composite_v4.csv", index_col=0) for w in WELLS}
GWC_TVDSS = 3797.0   # F-O4 log GWC 3818 m MD; drill floor 21.0 m above MSL (Mossgas 2000)
# depth-reference elevation above MSL (log depth datum = drill floor, from LIS/DLIS header EDF/APD unless noted)
KB = {"F-O1": 26.0, "F-O2": 25.0, "F-O3": 22.0, "F-S1": 22.0,
      "F-O4": 21.0,                   # Mossgas (2000) well data sheet: rotary table 21 m above MSL; GWC at 3818 m MD
      "F-O6": 26.45,                  # 25 m + 1.45 m TVD loss (wellpath survey)
      "F-O8": 25.0,
      "F-R1": 30.0}                   # SOEKOR (1980) completion report: KB 207 m above sea floor, water depth 177 m
GWC_MD = {w: GWC_TVDSS + KB[w] for w in WELLS}
RW77, CEC = 0.2678, 1.0
pd.Series(GWC_MD, name="GWC (m MD)")

## Step 1 — check Rw in the water legs
Below the contact, clean porous sand should be at Sw = 1, so Rwa = Rt·φ^m should sit on Rw (m = 2.1, the SCAL m*; Step 2 then calibrates m* in situ). The lower part of the Rwa
distribution (P10) in clean (VSH_GR < 0.25), porous (PHIT > 5 %), good-hole sand below the contact is compared with the SCAL brine.

In [ ]:
# Cell 3
rows = []
for w in WELLS:
    d = logs[w]; rt = st.RT_CURVE[w]; z = d.index.to_numpy()
    s = d[(d.index > GWC_MD[w]) & (d.VSH_GR < 0.25) & (d.PHIT > 0.05) & (d.BADHOLE != 1) & d[rt].notna()]
    if len(s) < 20: continue
    rwa = s[rt] * s.PHIT ** 2.1; rw_t = st.rw_at_T(RW77, 77, st.temperature_F(s.index.to_numpy()))
    rows.append(dict(well=w, n=len(s), from_m=round(s.index.min()), to_m=round(s.index.max()), Rwa_P10=round(rwa.quantile(0.1), 3),
                     Rwa_P50=round(rwa.median(), 3), Rw_SCAL_at_T=round(float(np.median(rw_t)), 3)))
rwcheck = pd.DataFrame(rows); rwcheck.to_csv(OUT_DIR / "rw_check_water_legs.csv", index=False); rwcheck

The P10 of Rwa in the water legs sits on, or within ~0.02 ohm·m of, the SCAL brine corrected to temperature (≈ 0.068 ohm·m);
the median is higher because of residual shale, thin beds and invasion. → **Rw = 0.2678 ohm·m @ 77 °F, temperature-corrected, is used**, with ±15 %
in the uncertainty cases.

## Step 2 — calibrate the cementation exponent in situ
With the SCAL parameters (m* 2.0–2.2, measured at room conditions without stress), clean sands **below** the contact compute
Sw ≈ 0.6–0.75 instead of 1.0 in every well. Raising Rw cannot fix this without a brine 2–3× fresher than the SCAL formation
brine. The consistent explanation is a higher in-situ m* in these tight, cemented sands (the F-O6 SCAL itself reports strong
stress effects). So m* is solved, with Rw fixed, such that the **median water-leg Sw = 1** (n* = 2.4).

In [ ]:
# Cell 4 — temperature, Rw, Qv for every well, then m* from the water legs
for w in WELLS:
    d = logs[w]; z = d.index.to_numpy(); T = st.temperature_F(z); rw = st.rw_at_T(RW77, 77, T); phi = d.PHIT.clip(lower=0.01)
    d["TEMP_F"] = T; d["RW"] = rw; d["QV"] = st.qv_from_cec(CEC, phi).clip(upper=5).where(d.PHIT.notna()); d["B_WS"] = st.b_juhasz(T, rw)
rows, pool = [], []
for w in WELLS:
    d = logs[w]; rt = st.RT_CURVE[w]
    s = d[(d.index > GWC_MD[w] + 5) & (d.VSH_GR < 0.25) & (d.PHIT > 0.04) & (d.BADHOLE != 1) & d[rt].notna()]
    if len(s) < 30: continue
    m = st.calibrate_m_star(s[rt], s.PHIT, s.RW, s.QV, s.B_WS, 2.4); rows.append(dict(well=w, n=len(s), m_star=m))
    pool.append(s.assign(RT=s[rt]))
P = pd.concat(pool); M_FIELD = st.calibrate_m_star(P.RT, P.PHIT, P.RW, P.QV, P.B_WS, 2.4)
mcal = pd.DataFrame(rows + [dict(well="FIELD", n=len(P), m_star=M_FIELD)]); mcal.to_csv(OUT_DIR / "m_star_water_leg_calibration.csv", index=False); mcal

## Step 3 — water saturation cases

In [ ]:
# Cell 5
CASES = {"BASE": dict(m=M_FIELD,       n=2.40),   # in-situ m* (water legs), mean SCAL n*
         "LOW":  dict(m=M_FIELD - 0.1, n=1.91),   # optimistic: m* -0.1 (well spread), F-O2 n*
         "HIGH": dict(m=M_FIELD + 0.1, n=2.88),   # pessimistic: m* +0.1, F-O6 n*
         "SCAL": dict(m=2.10,          n=2.40)}   # room-condition SCAL m* (for comparison: fails the water-leg test)
for w in WELLS:
    d = logs[w]; rt = d[st.RT_CURVE[w]]; phi = d.PHIT.clip(lower=0.01); ok = d.PHIT.notna() & rt.notna()
    for k, p in CASES.items():
        d[f"SW_WS_{k}"] = pd.Series(st.waxman_smits_sw(rt, phi, d.RW, d.QV.fillna(0), d.B_WS, p["m"], p["n"]), index=d.index).where(ok)
    d["SW_AR"] = pd.Series(st.archie_sw(rt, phi, d.RW, 1.0, 2.31, 2.4), index=d.index).where(ok)   # Archie, water-leg m = 2.31
    d["SW"] = d.SW_WS_BASE; d["BVW"] = d.SW * d.PHIT
chk = pd.DataFrame([dict(well=w, **{k: round(float(logs[w][f"SW_WS_{k}"][(logs[w].index > GWC_MD[w] + 5) & (logs[w].VSH_GR < 0.25) & (logs[w].PHIT > 0.04)].median()), 2)
                                    for k in CASES}) for w in WELLS])
print(f"m* (field) = {M_FIELD}; median Sw in the water legs per case:"); chk

## Step 4 — check against the well tests
No representative core Sw exists (cores cut with water-based mud), so the test is the DSTs: intervals that flowed dry gas
should show gas-zone saturations.

In [ ]:
# Cell 6
DST = [("F-O2", 3627, 3661, "DST: 31.5 MMscf/d dry gas"),
       ("F-O4", 3715, 3784, "DST #2 zones 2A+2B: 3.3-3.5 MMscf/d gas"),
       ("F-O4", 3806, 3811, "DST #1 zone 1: 0.55-0.92 MMscf/d gas (near GWC 3818 m)")]
rows = []
for w, t, b, note in DST:
    s = logs[w].loc[t:b]; net = s[(s.PHIT >= 0.06)]
    rows.append(dict(well=w, interval=f"{t}-{b}", test=note, PHIT_mean=round(net.PHIT.mean(), 3),
                     **{f"Sw_{k.lower()}": round(net[f"SW_WS_{k}"].mean(), 2) for k in CASES}, Sw_archie=round(net.SW_AR.mean(), 2)))
dstcheck = pd.DataFrame(rows); dstcheck.to_csv(OUT_DIR / "sw_vs_dst.csv", index=False); dstcheck

## Step 5 — saturation profile against depth (all wells, clean sand): base vs SCAL parameters

In [ ]:
# Cell 7
f, ax = plt.subplots(1, 8, figsize=(16, 6), sharey=True)
for a, w in zip(ax, WELLS):
    d = logs[w]; s = d[(d.VSH_GR < 0.35) & (d.PHIT > 0.04)]
    a.scatter(s.SW_WS_SCAL, s.index - KB[w], s=2, alpha=0.25, color="0.6", label="SCAL m*")
    a.scatter(s.SW, s.index - KB[w], s=2, alpha=0.5, label="base (in-situ m*)")
    a.axhline(GWC_TVDSS, color="#eb6834", lw=1); a.set_xlim(0, 1); a.set_title(w, fontsize=9); a.grid(alpha=0.3)
ax[0].invert_yaxis(); ax[0].set_ylabel("approx. TVDSS (m)"); ax[0].set_ylim(4060, 3560)
for a in ax: a.set_xlabel("Sw", fontsize=8)
ax[0].legend(fontsize=7, loc="lower left", markerscale=4)
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_sw_vs_depth.png", dpi=150); plt.show()

## Step 6 — cut-offs and net pay
* **Porosity**: the porosity at which the well's core transform gives 0.1 mD (Notebook 03): F-O1 6.3 %, F-O2 9.6 %, F-O3 9.3 %,
  F-O4 6.5 %, F-O6 9.4 %, F-S1 9.8 %; field 8.7 % for F-O8 and F-R1. (SOEKOR 1993 used 8 % in F-O2 and 6.5 % in F-O1.)
* **Shale**: VSH_GR ≤ 0.40.  **Saturation**: Sw ≤ 0.65 for pay — set by the well tests: F-O4 DST #2 flowed 3.4 MMscf/d of gas
  (with low-chloride, largely condensed water) from rock whose mean base-case Sw is 0.64; a 0.60 cut-off would call proven
  gas-flowing rock non-pay. Cell 9 shows net pay at 0.60 and 0.70.  **Interval**: top of reservoir window to the base of the
  window or the gas–water contact, whichever is shallower. Bad-hole samples are excluded (PHIT is null there).

In [ ]:
# Cell 8
PHI_CUT = {"F-O1": 0.063, "F-O2": 0.096, "F-O3": 0.093, "F-O4": 0.065, "F-O6": 0.094, "F-S1": 0.098}; PHI_FIELD = 0.087
VSH_CUT, SW_CUT, STEP = 0.40, 0.65, 0.1524
def summarise(w, sw_col="SW", phi_cut=None, sw_cut=None):
    swc = SW_CUT if sw_cut is None else sw_cut
    d = logs[w]; top, base = pm.WINDOWS[w]; base = min(base, GWC_MD[w])
    if base <= top:   # whole reservoir window below the contact (F-R1)
        return dict(well=w, top_m=top, base_m=pm.WINDOWS[w][1], gross_m=0.0, phi_cut=phi_cut or PHI_CUT.get(w, PHI_FIELD), net_res_m=0.0,
                    net_pay_m=0.0, NTG_pay=0.0, phi_pay=np.nan, sw_pay=np.nan, phih_m=0.0, hcpv_m=0.0, k_pay_mD_geo=np.nan,
                    note=f"reservoir top {top} m below GWC {GWC_MD[w]:.0f} m MD: water-bearing")
    s = d.loc[top:base]
    pc = phi_cut if phi_cut is not None else PHI_CUT.get(w, PHI_FIELD)
    res = (s.PHIT >= pc) & (s.VSH_GR <= VSH_CUT); pay = res & (s[sw_col] <= swc)
    def avg(x, m, wgt=None): return float(np.average(x[m], weights=(wgt[m] if wgt is not None else None))) if m.sum() else np.nan
    return dict(well=w, top_m=top, base_m=round(base, 1), gross_m=round(base - top, 1), phi_cut=pc,
                net_res_m=round(res.sum() * STEP, 1), net_pay_m=round(pay.sum() * STEP, 1), NTG_pay=round(pay.sum() * STEP / (base - top), 2),
                phi_pay=round(avg(s.PHIT, pay), 3), sw_pay=round(avg(s[sw_col], pay, s.PHIT), 2),
                phih_m=round(float((s.PHIT[pay]).sum() * STEP), 2), hcpv_m=round(float((s.PHIT * (1 - s[sw_col]))[pay].sum() * STEP), 2),
                k_pay_mD_geo=round(float(10 ** np.log10(s.PERM[pay].clip(lower=1e-4)).mean()), 3) if pay.sum() else np.nan,
                note="base = GWC" if base < pm.WINDOWS[w][1] else "base = M1 / window base")
pay = pd.DataFrame([summarise(w) for w in WELLS]); pay.to_csv(OUT_DIR / "net_pay_base.csv", index=False); pay

In [ ]:
# Cell 9 — how much do the answers move with the saturation parameters? (HCPV = sum phi (1-Sw) h over pay)
sens = []
for w in WELLS:
    row = dict(well=w)
    for k in ("LOW", "BASE", "HIGH", "SCAL"):
        r = summarise(w, f"SW_WS_{k}"); row[f"net_pay_{k.lower()}_m"] = r["net_pay_m"]; row[f"hcpv_{k.lower()}_m"] = r["hcpv_m"]
    r = summarise(w, "SW_AR"); row["net_pay_archie_m"] = r["net_pay_m"]; row["hcpv_archie_m"] = r["hcpv_m"]
    for c in (0.60, 0.70): row[f"net_pay_base_swcut{int(c*100)}_m"] = summarise(w, "SW", sw_cut=c)["net_pay_m"]
    sens.append(row)
sens = pd.DataFrame(sens); sens.to_csv(OUT_DIR / "net_pay_sensitivity.csv", index=False); sens

In [ ]:
# Cell 10 — write v5 (v4 + TEMP_F, RW, QV, SW, SW_WS_LOW/BASE/HIGH, SW_AR, BVW, NETRES, NETPAY flags)
NEWU = dict(TEMP_F="degF", RW="ohm.m", QV="meq/ml", B_WS="", SW="V/V", SW_WS_LOW="V/V", SW_WS_BASE="V/V", SW_WS_HIGH="V/V", SW_WS_SCAL="V/V", SW_AR="V/V", BVW="V/V", NETRES="", NETPAY="")
for w in WELLS:
    d = logs[w]; pc = PHI_CUT.get(w, PHI_FIELD); top, base = pm.WINDOWS[w]; inwin = (d.index >= top) & (d.index <= min(base, GWC_MD[w]))
    res = (d.PHIT >= pc) & (d.VSH_GR <= VSH_CUT) & inwin
    d["NETRES"] = res.astype(float).where(d.PHIT.notna()); d["NETPAY"] = (res & (d.SW <= SW_CUT)).astype(float).where(d.PHIT.notna())
    units = {c.mnemonic: c.unit for c in lasio.read(str(V4_DIR / f"{w}_composite_v4.las")).curves}; units.update(NEWU)
    d.to_csv(OUT_DIR / f"{w}_composite_v5.csv", float_format="%.5f")
    las = lasio.LASFile(); las.well["WELL"].value = w; las.well["FLD"].value = "F-O Gas Field"
    las.other = (f"{w} composite v5 (Notebook 04). Rt={st.RT_CURVE[w]}; Rw 0.2678@77F Arps-corrected; WS base m* {M_FIELD} (water-leg calibrated) n* 2.4, CEC 1 meq/100g; "
                 f"cut-offs PHIT>={pc}, VSH_GR<=0.4, SW<={SW_CUT}; GWC {GWC_MD[w]:.0f} m MD.")
    las.append_curve("DEPT", d.index.to_numpy(), unit="M")
    for c in d.columns: las.append_curve(c, d[c].to_numpy(), unit=units.get(c, ""))
    las.write(str(OUT_DIR / f"{w}_composite_v5.las"), version=2.0, fmt="%.5f")
print("written", len(WELLS), "wells")

## Notes for the paper
* State Rw source (SCAL brine) and its log check, then the in-situ m* calibration (8 wells give 2.29–2.55, field 2.44 vs
  room-condition SCAL 2.0–2.2). Using the SCAL m* would put the water legs at Sw ≈ 0.7 and overstate gas saturation.
* The F-O2/F-O6 spread in n* (1.9 vs 2.9) is the largest remaining Sw uncertainty — Cell 9 shows how much net pay and HCPV move.
* High-Sw sands well above the field GWC in F-O1, F-O2, F-O3 and F-S1 (Step 5 figure): long capillary transition in tight rock,
  or perched water / compartments — the question for Paper 2 (RFT pressures + saturation-height modelling).
* The CEC result (no correlation with Vsh) is itself a finding: conductivity is controlled by diagenetic pore-lining clay.
* RFT supercharging prevented a free-water level in F-O4; the contact used is the log GWC. Saturation-height modelling with the
  F-O6/F-O2 capillary-pressure data (Paper 2) will test the log Sw independently.